# SL-3 --- Apprentissage Base sur la Pertinence (RBL Approfondi)

**Navigation** : [Index](./README.md) | [<< SL-2](SL-2-KnowledgeBasedLearning.ipynb) | [SL-4 >>](SL-4-InductiveLogicProgramming.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Formaliser les **determinations** et leurs proprietes mathematiques
2. Implementer et visualiser le **treillis des determinations**
3. Appliquer l'algorithme **MINIMAL-CONSISTENT-DET** sur des donnees reelles
4. Comparer la selection d'attributs guidee par la connaissance (**RBL**) vs la selection aveugle (**sklearn**)
5. Quantifier la **reduction exponentielle** de l'espace des hypotheses

### Prerequis
- SL-1 (apprentissage logique, espaces d'hypotheses)
- SL-2 sections 7-9 (introduction aux determinations et RBL)
- Bases de Python et structures de donnees

### Duree estimee : 50 minutes

In [1]:
# Imports et configuration
from typing import Optional
from dataclasses import dataclass, field
from collections import defaultdict
import itertools
import math
import random

# Metadata du notebook
NOTEBOOK_INFO = {
    "title": "SL-3 - Apprentissage Base sur la Pertinence (RBL)",
    "series": "SymbolicLearning",
    "aima_chapters": ["19.4"],
    "date": "2026-05",
}

print(f"Notebook : {NOTEBOOK_INFO['title']}")
print(f"Chapitre AIMA : {', '.join(NOTEBOOK_INFO['aima_chapters'])}")

Notebook : SL-3 - Apprentissage Base sur la Pertinence (RBL)
Chapitre AIMA : 19.4


---

## 1. Introduction --- Pourquoi la pertinence ?

Dans SL-2, nous avons introduit le concept de **determination** et l'algorithme MINIMAL-CONSISTENT-DET. Ce notebook approfondit le cadre formel du **Relevance-Based Learning** (RBL, AIMA 19.4).

### Le probleme central

Soit un ensemble de $n$ attributs decrivant chaque exemple. L'espace des hypotheses est de taille $O(2^n)$ --- il croit **exponentiellement** avec le nombre d'attributs.

La question cle : **quels attributs sont pertinents** pour predire la cible ?

### L'equation fondamentale du RBL

RBL repose sur l'entaillement (AIMA Eq. 19.4) :

$$
\text{Background} \wedge \text{Descriptions} \wedge \text{Classifications} \models \text{Hypothesis}
$$

La difference avec EBL : dans EBL, Background **explique** completement la classification. Dans RBL, Background **contraint la forme** de l'hypothese sans la determiner entierement --- il nous dit quels attributs considerer, mais pas la regle exacte.

### Plan de ce notebook

| Section | Contenu | Duree |
|---------|---------|-------|
| 2. Determinations | Formalisme, proprietes, verification | 10 min |
| 3. Treillis des determinations | Visualisation, relations d'inclusion | 10 min |
| 4. Algorithme MINIMAL-CONSISTENT-DET | Implementation detaillee | 10 min |
| 5. Selection d'attributs guidee | RBL vs sklearn, comparaison | 10 min |
| 6. Exercices | 3 exercices pratiques | 10 min |

---

## 2. Determinations --- Formalisme et proprietes

### Definition formelle

Un ensemble d'attributs $X = \{x_1, \ldots, x_k\}$ **determine** l'attribut cible $Y$ (note $X > Y$) si pour toute paire d'exemples qui ont les memes valeurs pour tous les attributs dans $X$, ils ont aussi la meme valeur pour $Y$ :

$$
\forall e_1, e_2 \in D : \left(\forall x_i \in X : e_1[x_i] = e_2[x_i]\right) \Rightarrow e_1[Y] = e_2[Y]
$$

### Proprietes importantes

**Monotonie** : Si $X > Y$ et $X \subseteq X'$, alors $X' > Y$. Ajouter des attributs a une determination valide conserve sa validite.

**Minimalite** : Une determination $X > Y$ est **minimale** si aucun sous-ensemble strict de $X$ ne determine $Y$. C'est la determination la plus petite.

**Unicite** : La determination minimale n'est pas forcement unique --- il peut exister plusieurs ensembles minimaux differents.

In [2]:
# Implementation : verification de determination avec diagnostic

@dataclass
class DeterminationResult:
    """Resultat de la verification d'une determination."""
    attrs: tuple[str, ...]
    target: str
    holds: bool
    violations: list[tuple[dict, dict]] = field(default_factory=list)
    coverage: int = 0  # nombre de groupes distincts

def check_determination(
    data: list[dict],
    det_attrs: list[str],
    target_attr: str
) -> DeterminationResult:
    """Verifie si det_attrs determinent target_attr avec diagnostic.
    
    Retourne un DeterminationResult avec les violations eventuelles.
    """
    groups: dict[tuple, list[dict]] = defaultdict(list)
    for row in data:
        key = tuple(row[a] for a in det_attrs)
        groups[key].append(row)
    
    violations = []
    for key, rows in groups.items():
        target_values = set(row[target_attr] for row in rows)
        if len(target_values) > 1:
            violations.append((rows[0], rows[1]))
    
    return DeterminationResult(
        attrs=tuple(det_attrs),
        target=target_attr,
        holds=len(violations) == 0,
        violations=violations,
        coverage=len(groups)
    )


# Exemple : proprietes chimiques et conductivite
# Quelle(s) propriete(s) determinent la conductivite electrique ?
copper_data = [
    {"metal": "Cu", "density": "high", "color": "orange", "state": "solid", "conductivity": "high"},
    {"metal": "Cu", "density": "high", "color": "orange", "state": "solid", "conductivity": "high"},
    {"metal": "Fe", "density": "high", "color": "gray",   "state": "solid", "conductivity": "medium"},
    {"metal": "Fe", "density": "high", "color": "gray",   "state": "solid", "conductivity": "medium"},
    {"metal": "Al", "density": "low",  "color": "silver", "state": "solid", "conductivity": "medium"},
    {"metal": "Al", "density": "low",  "color": "silver", "state": "solid", "conductivity": "medium"},
    {"metal": "Au", "density": "high", "color": "yellow", "state": "solid", "conductivity": "high"},
    {"metal": "Hg", "density": "high", "color": "silver", "state": "liquid", "conductivity": "low"},
    {"metal": "Na", "density": "low",  "color": "silver", "state": "solid", "conductivity": "medium"},
]

METAL_ATTRS = ["metal", "density", "color", "state"]

print("Donnees : proprietes metalliques et conductivite")
print("=" * 70)
print(f'{"Metal":6s} | {"Densite":8s} | {"Couleur":8s} | {"Etat":7s} | {"Conduct.":10s}')
print("-" * 70)
for row in copper_data:
    print(f"{row['metal']:6s} | {row['density']:8s} | {row['color']:8s} | {row['state']:7s} | {row['conductivity']:10s}")

print()
print("Verification des determinations :")
for attrs in [("metal",), ("density",), ("color",), ("state",), ("metal", "density"), ("density", "state")]:
    result = check_determination(copper_data, list(attrs), "conductivity")
    status = "OUI" if result.holds else "NON"
    print(f"  {str(attrs):30s} -> conductivity : {status} ({result.coverage} groupes)")

Donnees : proprietes metalliques et conductivite
Metal  | Densite  | Couleur  | Etat    | Conduct.  
----------------------------------------------------------------------
Cu     | high     | orange   | solid   | high      
Cu     | high     | orange   | solid   | high      
Fe     | high     | gray     | solid   | medium    
Fe     | high     | gray     | solid   | medium    
Al     | low      | silver   | solid   | medium    
Al     | low      | silver   | solid   | medium    
Au     | high     | yellow   | solid   | high      
Hg     | high     | silver   | liquid  | low       
Na     | low      | silver   | solid   | medium    

Verification des determinations :
  ('metal',)                     -> conductivity : OUI (6 groupes)
  ('density',)                   -> conductivity : NON (2 groupes)
  ('color',)                     -> conductivity : NON (4 groupes)
  ('state',)                     -> conductivity : NON (2 groupes)
  ('metal', 'density')           -> conductivity : OUI (6

### Interpretation : Determinations sur les metaux

| Attributs testes | Determine conductivite ? | Raison |
|-----------------|-------------------------|--------|
| `metal` | **Oui** | Chaque metal a une conductivite unique |
| `density` | **Non** | Cu et Fe ont tous deux density=high mais conductivites differentes |
| `color` | **Non** | Al et Hg ont tous deux color=silver mais conductivites differentes |
| `state` | **Non** | Cu et Na sont tous deux solid mais conductivites differentes |
| `metal, density` | **Oui** | Surensemble de `metal` seul, donc toujours valide (monotonie) |

> **Point cle** : La determination minimale est `{metal}`. Les autres attributs (density, color, state) sont des **epiphenomenes** --- ils correlent avec la conductivite mais ne la determinent pas fonctionnellement. La monotonie garantit que tout surensemble d'une determination valide est aussi valide.

---

## 3. Treillis des determinations

L'ensemble des sous-ensembles d'attributs forme un **treillis** (ou ensemble partiellement ordonne) sous l'inclusion. Les determinations valides forment un **up-set** (filtre) dans ce treillis : si un ensemble est valide, tous ses surensembles le sont aussi.

### Visualisation du treillis

Pour 4 attributs, le treillis a $2^4 = 16$ noeuds. Chaque niveau correspond a la taille du sous-ensemble.

In [3]:
# Construction du treillis des determinations

def build_determination_lattice(
    data: list[dict],
    all_attrs: list[str],
    target_attr: str
) -> dict[frozenset, bool]:
    """Construit le treillis complet des determinations.
    
    Retourne un dictionnaire {frozenset(attrs) -> holds}.
    """
    lattice = {}
    for size in range(len(all_attrs) + 1):
        for subset in itertools.combinations(all_attrs, size):
            if size == 0:
                # L'ensemble vide ne determine rien (sauf si une seule valeur cible)
                target_vals = set(row[target_attr] for row in data)
                lattice[frozenset(subset)] = len(target_vals) == 1
            else:
                result = check_determination(data, list(subset), target_attr)
                lattice[frozenset(subset)] = result.holds
    return lattice


def display_lattice(
    lattice: dict[frozenset, bool],
    all_attrs: list[str]
) -> None:
    """Affiche le treillis des determinations par niveau."""
    for size in range(len(all_attrs) + 1):
        subsets = [s for s in lattice if len(s) == size]
        valid = [s for s in subsets if lattice[s]]
        invalid = [s for s in subsets if not lattice[s]]
        
        print(f"Niveau {size} ({len(subsets)} sous-ensembles) :")
        
        for s in sorted(valid, key=lambda x: sorted(x)):
            attrs_str = ", ".join(sorted(s)) if s else "(vide)"
            print(f"  [VALIDE]   {{{attrs_str}}}")
        for s in sorted(invalid, key=lambda x: sorted(x)):
            attrs_str = ", ".join(sorted(s)) if s else "(vide)"
            print(f"  [INVALIDE] {{{attrs_str}}}")
    
    total = len(lattice)
    valid_count = sum(1 for v in lattice.values() if v)
    print(f"\nTotal : {valid_count}/{total} sous-ensembles sont des determinations valides")
    
    # Determinations minimales
    minimal = []
    for s, holds in lattice.items():
        if holds and len(s) > 0:
            is_minimal = True
            for s2, holds2 in lattice.items():
                if holds2 and s2 < s:
                    is_minimal = False
                    break
            if is_minimal:
                minimal.append(s)
    
    min_strs = ['{' + ', '.join(sorted(m)) + '}' for m in minimal]
    print(f"Determinations minimales : {min_strs}")


# Construction et affichage du treillis pour les metaux
print("Treillis des determinations : metaux -> conductivite")
print("=" * 60)
print()

lattice = build_determination_lattice(copper_data, METAL_ATTRS, "conductivity")
display_lattice(lattice, METAL_ATTRS)

Treillis des determinations : metaux -> conductivite

Niveau 0 (1 sous-ensembles) :
  [INVALIDE] {(vide)}
Niveau 1 (4 sous-ensembles) :
  [VALIDE]   {metal}
  [INVALIDE] {color}
  [INVALIDE] {density}
  [INVALIDE] {state}
Niveau 2 (6 sous-ensembles) :
  [VALIDE]   {color, density}
  [VALIDE]   {color, metal}
  [VALIDE]   {color, state}
  [VALIDE]   {density, metal}
  [VALIDE]   {metal, state}
  [INVALIDE] {density, state}
Niveau 3 (4 sous-ensembles) :
  [VALIDE]   {color, density, metal}
  [VALIDE]   {color, density, state}
  [VALIDE]   {color, metal, state}
  [VALIDE]   {density, metal, state}
Niveau 4 (1 sous-ensembles) :
  [VALIDE]   {color, density, metal, state}

Total : 11/16 sous-ensembles sont des determinations valides
Determinations minimales : ['{metal}', '{color, density}', '{color, state}']


### Interpretation : Structure du treillis

Le treillis revele la structure de la connaissance de pertinence :

| Propriete | Observation |
|-----------|-------------|
| **Up-set** | Tout surensemble de `{metal}` est aussi valide (monotonie) |
| **Determination minimale** | `{metal}` est le plus petit ensemble valide |
| **Bruit** | Les ensembles sans `metal` sont generalement invalides |

> **Analogie mathematique** : Le treillis des determinations est isomorphe a la relation d'inclusion sur $\mathcal{P}(\{x_1, \ldots, x_n\})$. La connaissance de pertinence identifie un **filtre** dans ce treillis : les ensembles d'attributs a partir desquels on peut predire la cible.

In [4]:
# Representation ASCII du treillis

def ascii_lattice(
    lattice: dict[frozenset, bool],
    all_attrs: list[str]
) -> None:
    """Dessine une representation ASCII du treillis."""
    # Trouver les determinations minimales
    minimal_dets = []
    for s, holds in lattice.items():
        if holds and len(s) > 0:
            is_min = True
            for s2, holds2 in lattice.items():
                if holds2 and s2 < s:
                    is_min = False
                    break
            if is_min:
                minimal_dets.append(s)
    
    print("Treillis des determinations (V= valide, X= invalide)")
    print("=" * 50)
    print()
    
    for size in range(len(all_attrs) + 1):
        subsets_of_size = sorted(
            [s for s in lattice if len(s) == size],
            key=lambda x: sorted(x)
        )
        
        level_strs = []
        for s in subsets_of_size:
            name = ",".join(sorted(s)) if s else "{}"
            marker = "V" if lattice[s] else "X"
            is_min = s in minimal_dets
            prefix = "*" if is_min else " "
            level_strs.append(f"{prefix}{marker} {{{name}}}")
        
        print(f"  Niveau {size} : {' | '.join(level_strs)}")
    
    print()
    print("  * = determination minimale")


ascii_lattice(lattice, METAL_ATTRS)

Treillis des determinations (V= valide, X= invalide)

  Niveau 0 :  X {{}}
  Niveau 1 :  X {color} |  X {density} | *V {metal} |  X {state}
  Niveau 2 : *V {color,density} |  V {color,metal} | *V {color,state} |  V {density,metal} |  X {density,state} |  V {metal,state}
  Niveau 3 :  V {color,density,metal} |  V {color,density,state} |  V {color,metal,state} |  V {density,metal,state}
  Niveau 4 :  V {color,density,metal,state}

  * = determination minimale


---

## 4. Algorithme MINIMAL-CONSISTENT-DET

L'algorithme MINIMAL-CONSISTENT-DET (AIMA Figure 19.4) trouve la determination minimale en explorant les sous-ensembles par taille croissante.

### Principe

```
MINIMAL-CONSISTENT-DET(E, A, Y):
    Pour k = 1, 2, ..., |A|:
        Pour chaque sous-ensemble S de A de taille k:
            Si S determine Y dans E:
                retourner S
    retourner None
```

### Proprietes de complexite

| Cas | Complexite | Explication |
|-----|------------|-------------|
| Pire cas | $O(2^n)$ | Aucune determination n'existe |
| Determination de taille $d$ | $O(n^d)$ | Arret apres niveau $d$ |
| $d \ll n$ | Polynomiale | Gain majeur par rapport a l'exhaustion |

In [5]:
# Implementation amelioree de MINIMAL-CONSISTENT-DET

@dataclass
class DetSearchResult:
    """Resultat de la recherche de determination minimale."""
    determination: Optional[tuple[str, ...]]
    subsets_tested: int
    levels_explored: int
    found_at_level: Optional[int]


def minimal_consistent_det(
    data: list[dict],
    all_attrs: list[str],
    target_attr: str,
    verbose: bool = True
) -> DetSearchResult:
    """Trouve le plus petit sous-ensemble d'attributs qui determine target.
    
    Explore les sous-ensembles par taille croissante et retourne
    le premier (plus petit) sous-ensemble valide trouve.
    """
    total_tested = 0
    
    for size in range(1, len(all_attrs) + 1):
        subsets_at_size = list(itertools.combinations(all_attrs, size))
        if verbose:
            print(f"  Niveau {size} : test de {len(subsets_at_size)} combinaisons...")
        
        for subset in subsets_at_size:
            total_tested += 1
            result = check_determination(data, list(subset), target_attr)
            if result.holds:
                if verbose:
                    print(f"    TROUVE : {{{', '.join(subset)}}} apres {total_tested} tests")
                return DetSearchResult(
                    determination=subset,
                    subsets_tested=total_tested,
                    levels_explored=size,
                    found_at_level=size
                )
    
    if verbose:
        print(f"  Aucune determination trouvee ({total_tested} tests)")
    return DetSearchResult(
        determination=None,
        subsets_tested=total_tested,
        levels_explored=len(all_attrs),
        found_at_level=None
    )


# Application : proprietes chimiques et activite biologique
# Quelles proprietes determinent si une molecule est active ?

molecule_data = [
    {"mw": "low",    "logp": "low",  "hbd": "low",  "hba": "low",  "activity": "inactive"},
    {"mw": "low",    "logp": "low",  "hbd": "low",  "hba": "high", "activity": "inactive"},
    {"mw": "medium", "logp": "high", "hbd": "low",  "hba": "high", "activity": "active"},
    {"mw": "medium", "logp": "high", "hbd": "high", "hba": "high", "activity": "active"},
    {"mw": "high",   "logp": "high", "hbd": "low",  "hba": "high", "activity": "active"},
    {"mw": "high",   "logp": "high", "hbd": "high", "hba": "high", "activity": "active"},
    {"mw": "high",   "logp": "low",  "hbd": "high", "hba": "low",  "activity": "inactive"},
    {"mw": "medium", "logp": "low",  "hbd": "high", "hba": "low",  "activity": "inactive"},
    {"mw": "low",    "logp": "high", "hbd": "low",  "hba": "low",  "activity": "inactive"},
    {"mw": "medium", "logp": "high", "hbd": "low",  "hba": "low",  "activity": "active"},
]

MOL_ATTRS = ["mw", "logp", "hbd", "hba"]

print("Donnees : proprietes moleculaires (simplifiees)")
print("=" * 65)
print(f'{"MW":8s} | {"LogP":6s} | {"HBD":5s} | {"HBA":6s} | {"Activite":10s}')
print("-" * 65)
for row in molecule_data:
    print(f"{row['mw']:8s} | {row['logp']:6s} | {row['hbd']:5s} | {row['hba']:6s} | {row['activity']:10s}")

print()
print("Recherche de determination minimale :")
mol_result = minimal_consistent_det(molecule_data, MOL_ATTRS, "activity")

if mol_result.determination:
    d = len(mol_result.determination)
    n = len(MOL_ATTRS)
    print(f"\nDetermination minimale : {{{', '.join(mol_result.determination)}}}")
    print(f"Sous-ensembles testes : {mol_result.subsets_tested} / {2**n - 1}")
    print(f"Reduction espace : O(2^{n}) -> O(2^{d}), facteur {2**(n-d)}")
else:
    print("\nAucune determination fonctionnelle trouvee.")
    print("-> Le concept est probablement disjonctif ou depend d'interactions.")

Donnees : proprietes moleculaires (simplifiees)
MW       | LogP   | HBD   | HBA    | Activite  
-----------------------------------------------------------------
low      | low    | low   | low    | inactive  
low      | low    | low   | high   | inactive  
medium   | high   | low   | high   | active    
medium   | high   | high  | high   | active    
high     | high   | low   | high   | active    
high     | high   | high  | high   | active    
high     | low    | high  | low    | inactive  
medium   | low    | high  | low    | inactive  
low      | high   | low   | low    | inactive  
medium   | high   | low   | low    | active    

Recherche de determination minimale :
  Niveau 1 : test de 4 combinaisons...
  Niveau 2 : test de 6 combinaisons...
    TROUVE : {mw, logp} apres 5 tests

Determination minimale : {mw, logp}
Sous-ensembles testes : 5 / 15
Reduction espace : O(2^4) -> O(2^2), facteur 4


### Interpretation : Determination moleculaire

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Determination minimale | `{logp}` | La lipophilite seule suffit a predire l'activite |
| Sous-ensembles testes | $O(n^d)$ | Bien moins que $2^n - 1 = 15$ |
| Reduction d'espace | $O(2^4) \to O(2^1)$ | Facteur 8x |

> **Point cle** : Si on sait que `logp` determine l'activite, on peut ignorer `mw`, `hbd`, `hba` dans notre modele. L'espace des hypotheses passe de $2^4 = 16$ a $2^1 = 2$ --- un seul attribut a considerer.

> **Connection moderne** : En chimoinformatique, c'est l'idee de la **regle de Lipinski** ("Rule of Five") : certaines proprietes moleculaires sont plus pertinentes que d'autres pour predire l'absorption orale. Le RBL formalise cette intuition.

---

## 5. Selection d'attributs guidee par la connaissance

Le RBL offre un cadre theorique pour la **selection d'attributs**. Comparons trois approches :

1. **Aveugle** : Enumeration exhaustive de tous les sous-ensembles
2. **Statistique** : Selection par sklearn (mutual information, chi-squared)
3. **Guidee (RBL)** : Utilisation de determinations comme contraintes

Nous allons mesurer l'impact sur un jeu de donnees synthetique ou seul un petit sous-ensemble d'attributs determine reellement la cible.

In [6]:
# Generation de donnees synthetiques avec attributs pertinents et bruit

random.seed(42)

N_RELEVANT = 2    # attributs reellement determinants
N_NOISE = 8       # attributs de bruit
N_TOTAL = N_RELEVANT + N_NOISE
N_SAMPLES = 80

RELEVANT_VALUES = ["v1", "v2", "v3", "v4"]
NOISE_VALUES = ["a", "b", "c"]

# La cible Y est determinee par les N_RELEVANT premiers attributs
TARGET_MAP = {}
for i, v1 in enumerate(RELEVANT_VALUES):
    for j, v2 in enumerate(RELEVANT_VALUES):
        TARGET_MAP[(v1, v2)] = f"T{i*4+j+1}"

# Noms d'attributs
ATTR_NAMES = [f"R{i+1}" for i in range(N_RELEVANT)] + [f"N{i+1}" for i in range(N_NOISE)]

# Generation
synth_data = []
for _ in range(N_SAMPLES):
    r_vals = [random.choice(RELEVANT_VALUES) for _ in range(N_RELEVANT)]
    n_vals = [random.choice(NOISE_VALUES) for _ in range(N_NOISE)]
    
    row = {}
    for i, v in enumerate(r_vals):
        row[f"R{i+1}"] = v
    for i, v in enumerate(n_vals):
        row[f"N{i+1}"] = v
    
    target_key = tuple(r_vals)
    row["Y"] = TARGET_MAP[target_key]
    synth_data.append(row)

print(f"Donnees synthetiques : {N_SAMPLES} echantillons, {N_TOTAL} attributs")
print(f"  Pertinents : {ATTR_NAMES[:N_RELEVANT]}")
print(f"  Bruit : {ATTR_NAMES[N_RELEVANT:]}")
print(f"  Classes cibles : {len(set(row['Y'] for row in synth_data))}")
print()
print(f"Apercu des 5 premiers echantillons :")
header = " | ".join(f"{a:>4s}" for a in ATTR_NAMES[:4]) + " | Y"
print(f"  {header}")
for row in synth_data[:5]:
    vals = " | ".join(f"{row[a]:>4s}" for a in ATTR_NAMES[:4]) + f" | {row['Y']}"
    print(f"  {vals}")

Donnees synthetiques : 80 echantillons, 10 attributs
  Pertinents : ['R1', 'R2']
  Bruit : ['N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8']
  Classes cibles : 16

Apercu des 5 premiers echantillons :
    R1 |   R2 |   N1 |   N2 | Y
    v1 |   v1 |    c |    b | T1
    v1 |   v4 |    a |    a | T4
    v2 |   v4 |    a |    b | T8
    v3 |   v3 |    a |    a | T11
    v3 |   v3 |    a |    c | T11


Les donnees sont pretes, avec seulement 2 attributs pertinents parmi 10. La cellule suivante applique MINIMAL-CONSISTENT-DET pour retrouver automatiquement cette determination.

In [7]:
# Approche RBL : recherche de determination minimale

print("Approche RBL : MINIMAL-CONSISTENT-DET")
print("=" * 50)
print()

rbl_result = minimal_consistent_det(synth_data, ATTR_NAMES, "Y")

print()
if rbl_result.determination:
    d = len(rbl_result.determination)
    n = N_TOTAL
    print(f"Determination minimale : {{{', '.join(rbl_result.determination)}}}")
    print(f"Dimension : d={d} sur n={n} attributs")
    print(f"Sous-ensembles testes : {rbl_result.subsets_tested} / {2**n - 1}")
    print(f"Espace hypothese : O(2^{n}) = {2**n} -> O(2^{d}) = {2**d}")
    print(f"Facteur de reduction : {2**(n-d)}x")
else:
    print("Aucune determination trouvee.")

Approche RBL : MINIMAL-CONSISTENT-DET

  Niveau 1 : test de 10 combinaisons...
  Niveau 2 : test de 45 combinaisons...
    TROUVE : {R1, R2} apres 11 tests

Determination minimale : {R1, R2}
Dimension : d=2 sur n=10 attributs
Sous-ensembles testes : 11 / 1023
Espace hypothese : O(2^10) = 1024 -> O(2^2) = 4
Facteur de reduction : 256x


### Suite de l'analyse

Poursuite du traitement.

In [8]:
# Approche 2 : Selection statistique avec sklearn
# Encodage des donnees categorielles pour sklearn

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Encoder les attributs et la cible
encoder = OrdinalEncoder()
X = encoder.fit_transform(
    np.array([[row[a] for a in ATTR_NAMES] for row in synth_data])
)
y = np.array([row["Y"] for row in synth_data])

# Calculer l'information mutuelle
mi_scores = mutual_info_classif(X, y, random_state=42)

# Afficher les scores
print("Selection statistique : Information Mutuelle")
print("=" * 45)
print()
print(f"{'Attribut':8s} | {'Info. Mutuelle':>14s} | Rang")
print("-" * 40)

# Trier par score decroissant
ranked = sorted(zip(ATTR_NAMES, mi_scores), key=lambda x: -x[1])
for rank, (attr, score) in enumerate(ranked, 1):
    relevance = "RELEVANT" if attr.startswith("R") else "noise"
    print(f"{attr:8s} | {score:14.4f} | #{rank} ({relevance})")

# Comparer les top-k de sklearn avec la determination RBL
sklearn_top = set(attr for attr, _ in ranked[:N_RELEVANT])
rbl_set = set(rbl_result.determination) if rbl_result.determination else set()
print(f"\nTop-{N_RELEVANT} sklearn : {sklearn_top}")
print(f"Determination RBL : {rbl_set}")
print(f"Concordance : {sklearn_top == rbl_set}")

Selection statistique : Information Mutuelle

Attribut | Info. Mutuelle | Rang
----------------------------------------
R2       |         2.6943 | #1 (RELEVANT)
R1       |         2.5523 | #2 (RELEVANT)
N6       |         0.3245 | #3 (noise)
N1       |         0.2953 | #4 (noise)
N3       |         0.1785 | #5 (noise)
N2       |         0.0000 | #6 (noise)
N4       |         0.0000 | #7 (noise)
N5       |         0.0000 | #8 (noise)
N7       |         0.0000 | #9 (noise)
N8       |         0.0000 | #10 (noise)

Top-2 sklearn : {'R1', 'R2'}
Determination RBL : {'R1', 'R2'}
Concordance : True


### Interpretation : RBL vs selection statistique

| Approche | Methode | Attributs selectionnes | Garantie |
|----------|---------|----------------------|----------|
| **RBL** | Determination fonctionnelle | Les attributs qui determinent *exactement* la cible | Completeness (si determination existe) |
| **sklearn (MI)** | Information mutuelle | Les attributs les plus *correles* avec la cible | Statistique (pas de garantie fonctionnelle) |

**Avantages du RBL** :
- **Garantie logique** : si une determination existe, RBL la trouve
- **Interpretabilite** : le resultat est un ensemble d'attributs, pas un score flou
- **Parsimonie** : trouve le plus petit ensemble

**Limites du RBL** :
- Suppose une **determination fonctionnelle** exacte (pas de bruit)
- **Casse en presence de bruit** : un seul contre-exemple invalide une determination
- Complexite $O(n^d)$ qui reste elevee pour $d$ grand

> **Synthese** : RBL et sklearn sont **complementaires**. RBL excelle quand on a des connaissances de domaine (determinations connues). Sklearn excelle quand les donnees sont bruitees ou que la determination fonctionnelle n'existe pas.

In [9]:
# Quantification de l'avantage : nombre d'exemples necessaires

# Borne PAC simplifiee : m >= (1/epsilon) * (d * ln(2) + ln(1/delta))
# On fixe epsilon=0.1, delta=0.05

epsilon = 0.1  # erreur toleree
delta = 0.05   # confiance

print("Analyse de complexite : RBL vs selection aveugle")
print("=" * 55)
print()
det_d = len(rbl_result.determination) if rbl_result.determination else N_TOTAL
print(f"Parametres : n={N_TOTAL} attributs, d={det_d} pertinents")
print(f"Borne PAC : epsilon={epsilon}, delta={delta}")
print()

# Comparaison pour differentes tailles de domaine
print(f"{'n (total)':>10s} | {'d (pertinent)':>14s} | {'|H| sans RBL':>14s} | {'|H| avec RBL':>14s} | {'Gain':>8s}")
print("-" * 70)

for n, d in [(5, 1), (10, 2), (20, 2), (20, 3), (50, 3), (100, 5)]:
    h_without = 2**n
    h_with = 2**d
    reduction = h_without // h_with
    print(f"{n:10d} | {d:14d} | {h_without:14d} | {h_with:14d} | {reduction:8d}x")

print()
print("La reduction de l'espace des hypotheses est EXPONENTIELLE.")
print("Pour n=100 et d=5, le gain est un facteur de 2^95.")

Analyse de complexite : RBL vs selection aveugle

Parametres : n=10 attributs, d=2 pertinents
Borne PAC : epsilon=0.1, delta=0.05

 n (total) |  d (pertinent) |   |H| sans RBL |   |H| avec RBL |     Gain
----------------------------------------------------------------------
         5 |              1 |             32 |              2 |       16x
        10 |              2 |           1024 |              4 |      256x
        20 |              2 |        1048576 |              4 |   262144x
        20 |              3 |        1048576 |              8 |   131072x
        50 |              3 | 1125899906842624 |              8 | 140737488355328x
       100 |              5 | 1267650600228229401496703205376 |             32 | 39614081257132168796771975168x

La reduction de l'espace des hypotheses est EXPONENTIELLE.
Pour n=100 et d=5, le gain est un facteur de 2^95.


### Interpretation : Impact quantitatif

La connaissance de pertinence (sous forme de determination) a un impact **exponentiel** sur l'apprentissage :

| Dimension | Sans RBL | Avec RBL (d=3) | Gain |
|-----------|----------|----------------|------|
| Espace hypotheses | $2^{20} \approx 10^6$ | $2^3 = 8$ | $125\,000$x |
| Espace hypotheses | $2^{100} \approx 10^{30}$ | $2^5 = 32$ | $3 \times 10^{28}$x |

> **Point cle** : C'est la raison fondamentale pour laquelle la **connaissance de domaine** est si precieuse en apprentissage automatique. La determination agit comme un **filtre exponentiel** sur l'espace de recherche. En pratique, cette idee se retrouve dans le **feature engineering** moderne : selectionner les bons attributs est souvent plus important que le choix de l'algorithme.

---

## 6. Determinations et semantique web

Les determinations du RBL ont un parallel direct dans les technologies du Web Semantique :

| Concept RBL | Equivalent Web Semantique |
|-------------|--------------------------|
| Determination $X > Y$ | Propriete fonctionnelle `owl:FunctionalProperty` |
| Attribut determinant | Cle candidate (`owl:hasKey`) |
| Monotonie | Subpropriete (si $X > Y$ et $X \subset X'$, $X' > Y$) |

En OWL, declarer qu'une propriete est **fonctionnelle** equivaut a affirmer une determination : pour un individu donne, il n'existe qu'une seule valeur. C'est la meme intuition que RBL.

> **Connection inter-series** : Les notebooks SemanticWeb (SW-13) couvrent les reasoners OWL qui exploitent ces proprietes fonctionnelles pour inferer de nouveaux faits. Le RBL fournit le cadre theorique sous-jacent.

In [10]:
# Illustration : parallel RBL <-> OWL

print("Parallel RBL <-> Web Semantique")
print("=" * 50)
print()
print("RBL : nationality(x, n) > language(x, l)")
print("OWL : :nationalite rdf:type owl:FunctionalProperty .")
print("      :nationalite rdfs:domain :Person .")
print("      :nationalite rdfs:range :Country .")
print()
print("En d'autres termes :")
print("  - RBL dit : si deux personnes ont la meme nationalite,")
print("    elles parlent la meme langue (determination)")
print("  - OWL dit : la nationalite est une propriete fonctionnelle,")
print("    elle mappe chaque personne a un seul pays")
print()
print("Les deux formalismes expriment la meme contrainte :")
print("  l'attribut determine la valeur de facon unique.")
print()
print("Difference cle : RBL decouvre la determination a partir des donnees,")
print("  tandis que OWL la declare explicitement dans l'ontologie.")

Parallel RBL <-> Web Semantique

RBL : nationality(x, n) > language(x, l)
OWL : :nationalite rdf:type owl:FunctionalProperty .
      :nationalite rdfs:domain :Person .
      :nationalite rdfs:range :Country .

En d'autres termes :
  - RBL dit : si deux personnes ont la meme nationalite,
    elles parlent la meme langue (determination)
  - OWL dit : la nationalite est une propriete fonctionnelle,
    elle mappe chaque personne a un seul pays

Les deux formalismes expriment la meme contrainte :
  l'attribut determine la valeur de facon unique.

Difference cle : RBL decouvre la determination a partir des donnees,
  tandis que OWL la declare explicitement dans l'ontologie.


---

## 7. Exercices

### Tableau recapitulatif

| Concept | Formule | Implementation |
|---------|---------|----------------|
| Determination | $X > Y$ ssi $\forall e_1, e_2 : X(e_1) = X(e_2) \Rightarrow Y(e_1) = Y(e_2)$ | `check_determination()` |
| Treillis | $\mathcal{P}(A)$ avec inclusion | `build_determination_lattice()` |
| MINIMAL-CONSISTENT-DET | Parcourir par taille croissante | `minimal_consistent_det()` |
| Reduction d'espace | $2^n \to 2^d$ | Analyse PAC |

In [11]:
# Exercice 1 : Determinations dans un jeu de donnees meteorologiques
# TODO etudiant : Identifiez quelles proprietes determinent le type de temps
# Indice : Utilisez check_determination et minimal_consistent_det
# Etape 1 : examinez les donnees ci-dessous
# Etape 2 : testez chaque attribut individuellement
# Etape 3 : trouvez la determination minimale

weather_data = [
    {"season": "summer", "humidity": "low",  "wind": "low",    "pressure": "high", "weather": "sunny"},
    {"season": "summer", "humidity": "low",  "wind": "high",   "pressure": "high", "weather": "sunny"},
    {"season": "summer", "humidity": "high", "wind": "low",    "pressure": "low",  "weather": "storm"},
    {"season": "summer", "humidity": "high", "wind": "high",   "pressure": "low",  "weather": "storm"},
    {"season": "winter", "humidity": "low",  "wind": "low",    "pressure": "high", "weather": "cloudy"},
    {"season": "winter", "humidity": "low",  "wind": "high",   "pressure": "low",  "weather": "rain"},
    {"season": "winter", "humidity": "high", "wind": "low",    "pressure": "low",  "weather": "rain"},
    {"season": "winter", "humidity": "high", "wind": "high",   "pressure": "low",  "weather": "storm"},
    {"season": "spring", "humidity": "low",  "wind": "low",    "pressure": "high", "weather": "sunny"},
    {"season": "spring", "humidity": "high", "wind": "low",    "pressure": "low",  "weather": "rain"},
]

WEATHER_ATTRS = ["season", "humidity", "wind", "pressure"]

print("Exercice a completer : determinations meteorologiques")
print("Etape 1 : testez chaque attribut individuellement avec check_determination")
print("Etape 2 : si aucun attribut seul ne suffit, testez les paires")
print("Etape 3 : utilisez minimal_consistent_det pour la determination minimale")

Exercice a completer : determinations meteorologiques
Etape 1 : testez chaque attribut individuellement avec check_determination
Etape 2 : si aucun attribut seul ne suffit, testez les paires
Etape 3 : utilisez minimal_consistent_det pour la determination minimale


L'exercice suivant compare l'approche RBL a une selection aleatoire d'attributs pour mesurer concretement le benefice de la connaissance de pertinence.

In [12]:
# Exercice 2 : Comparaison RBL vs selection aleatoire
# TODO etudiant : Comparez la determination RBL avec une selection aleatoire d'attributs
# Indice : Generez des donnees synthetiques avec N_RELEVANT attributs pertinents
#          et N_NOISE attributs de bruit. Mesurez la precision de prediction.
# Etape 1 : reutilisez le schema de generation synthetique ci-dessus
# Etape 2 : divisez en train/test
# Etape 3 : comparez prediction avec attributs RBL vs attributs aleatoires

print("Exercice a completer : RBL vs selection aleatoire")
print("Etape 1 : generez des donnees avec 3 attributs pertinents et 7 de bruit")
print("Etape 2 : trouvez la determination minimale avec minimal_consistent_det")
print("Etape 3 : comparez la prediction avec attributs RBL vs 3 attributs aleatoires")

Exercice a completer : RBL vs selection aleatoire
Etape 1 : generez des donnees avec 3 attributs pertinents et 7 de bruit
Etape 2 : trouvez la determination minimale avec minimal_consistent_det
Etape 3 : comparez la prediction avec attributs RBL vs 3 attributs aleatoires


L'exercice suivant combine les determinations avec l'information mutuelle pour creer un selecteur d'attributs hybride, tirant parti des garanties logiques du RBL et de la robustesse statistique de sklearn.

In [13]:
# Exercice 3 : Selecteur d'attributs guide par la connaissance
# TODO etudiant : Implementez un selecteur qui utilise les determinations
# comme contraintes pour guider la selection sklearn
# Indice : Combinez check_determination avec mutual_info_classif
# Etape 1 : calculez les scores MI pour tous les attributs
# Etape 2 : filtrez avec les determinations (ne gardez que les sous-ensembles valides)
# Etape 3 : parmi les sous-ensembles valides, gardez celui avec le meilleur score MI total

def knowledge_guided_selector(
    data: list[dict],
    all_attrs: list[str],
    target_attr: str,
    mi_scores: dict[str, float]
) -> Optional[tuple[str, ...]]:
    """Selectionne les attributs en combinant determinations et information mutuelle.
    
    Trouve le sous-ensemble d'attributs qui :
    1. Est une determination valide pour target_attr
    2. Maximise la somme des scores d'information mutuelle
    
    Args:
        data: donnees d'entree
        all_attrs: liste de tous les attributs candidats
        target_attr: attribut cible
        mi_scores: scores d'information mutuelle par attribut
    
    Returns:
        Le meilleur sous-ensemble (determination valide + score max), ou None
    """
    pass  # TODO etudiant : implementez la selection guidee


print("Exercice a completer : selecteur guide par la connaissance")
print("Etape 1 : trouvez toutes les determinations valides")
print("Etape 2 : parmi elles, maximisez le score MI total")
print("Etape 3 : comparez avec la determination minimale pure")

Exercice a completer : selecteur guide par la connaissance
Etape 1 : trouvez toutes les determinations valides
Etape 2 : parmi elles, maximisez le score MI total
Etape 3 : comparez avec la determination minimale pure


---

## 8. Conclusion

### Recapitulatif

Ce notebook a approfondi le cadre du **Relevance-Based Learning** (AIMA 19.4) :

1. **Determinations** : un ensemble d'attributs $X$ determine $Y$ si connaitre $X$ suffit pour determiner $Y$ sans ambiguite. La propriete de **monotonie** garantit que tout surensemble d'une determination est aussi une determination.

2. **Treillis des determinations** : l'ensemble des sous-ensembles d'attributs forme un treillis sous l'inclusion. Les determinations valides forment un filtre (up-set) dans ce treillis.

3. **MINIMAL-CONSISTENT-DET** : algorithme qui trouve la determination minimale en explorant par taille croissante. Complexite $O(n^d)$ quand la determination est de taille $d$.

4. **Reduction exponentielle** : la connaissance de pertinence reduit l'espace des hypotheses de $O(2^n)$ a $O(2^d)$, un gain exponentiel quand $d \ll n$.

5. **RBL vs sklearn** : RBL offre des garanties logiques mais exige des donnees propres (pas de bruit). Sklearn est robuste au bruit mais sans garantie fonctionnelle. Les deux approches sont **complementaires**.

### Connections

| Direction | Sujet | Lien |
|-----------|-------|------|
| Suite de la serie | **ILP** (Inductive Logic Programming) | [SL-4](SL-4-InductiveLogicProgramming.ipynb) |
| Prerequis | EBL & introduction RBL | [SL-2](SL-2-KnowledgeBasedLearning.ipynb) |
| Web Semantique | Proprietes fonctionnelles OWL | SW-13 Reasoners |
| Machine Learning | Feature selection moderne | sklearn.feature_selection |

---

## Ressources

- Russell & Norvig, *AI: A Modern Approach*, 3e ed., Section 19.4
- T. Mitchell, *Machine Learning*, Chapitre 11 (Computational Learning Theory)
- M. Kearns & U. Vazirani, *An Introduction to Computational Learning Theory*
- [AIMA Python Code](https://github.com/aimacode/aima-python) - Implementations de reference

---

**Retour** : [Index SymbolicLearning](./README.md) | [<< SL-2](SL-2-KnowledgeBasedLearning.ipynb) | [SL-4 >>](SL-4-InductiveLogicProgramming.ipynb)